## Imports

In [42]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
import os
import matplotlib.pyplot as plt

from dotenv import load_dotenv
load_dotenv()

False

## Load clean Dataset

In [43]:
USE_S3 = os.getenv('USE_S3', 'false').lower() == 'true'
S3_BUCKET = os.getenv('S3_BUCKET', '')

BASE_DIR = os.path.dirname(os.path.abspath('__file__'))
DATA_DIR = os.path.join(BASE_DIR, '../data')

if USE_S3:
    df = pd.read_csv(f's3://{S3_BUCKET}/cleaned_data.csv')
else:
    df = pd.read_csv(os.path.join(DATA_DIR, 'cleaned_data.csv'))

print(f'Shape of the data : {df.shape}')
df.head()


Shape of the data : (13690540, 15)


,order_id,product_id,add_to_cart_order,reordered,product_name,aisle_id,department_id,aisle,department,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order,is_first_order
0,2,33120,1,1,Organic Egg Whites,86,16,eggs,dairy eggs,202279,3,5,9,8.0,0
1,2,28985,2,1,Michigan Organic Kale,83,4,fresh vegetables,produce,202279,3,5,9,8.0,0
2,2,9327,3,0,Garlic Powder,104,13,spices seasonings,pantry,202279,3,5,9,8.0,0
3,2,45918,4,1,Coconut Butter,19,13,oils vinegars,pantry,202279,3,5,9,8.0,0
4,2,30035,5,0,Natural Sweetener,17,13,baking ingredients,pantry,202279,3,5,9,8.0,0


## Features and Target

In [44]:
features = [
    "add_to_cart_order",
    "aisle_id",
    "department_id",
    "order_number",
    "order_dow",
    "order_hour_of_day",
    "days_since_prior_order",
    "user_id"
]

df_small = df.sample(1000000, random_state=0) 

X = df_small[features]
y = df_small["reordered"]

## Train Test Split

In [45]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Taille du Dataset d'entraînement : {X_train.shape}")
print(f"Taille du Dataset de test : {X_test.shape}")

Taille du Dataset d'entraînement : (800000, 8)
Taille du Dataset de test : (200000, 8)


## Feature engineering

### User features

In [ ]:
# Feature 1 
user_total_orders = (X_train.groupby("user_id")["order_number"].max())

X_train["user_total_orders"] = X_train["user_id"].map(user_total_orders)
X_test["user_total_orders"] = X_test["user_id"].map(user_total_orders)

# Feature 2 
user_avg_days_between_orders = (X_train.groupby("user_id")["days_since_prior_order"].mean())

X_train["user_avg_days_between_orders"] = X_train["user_id"].map(user_avg_days_between_orders)
X_test["user_avg_days_between_orders"] = X_test["user_id"].map(user_avg_days_between_orders)

X_train.head()

,add_to_cart_order,aisle_id,department_id,order_number,order_dow,order_hour_of_day,days_since_prior_order,user_id,user_total_orders,user_avg_days_between_orders
1226216,10,107,19,3,3,9,7.0,27171,15,9.333333
2209748,8,108,16,16,6,14,6.0,47138,16,15.500000
4695710,1,78,19,3,3,9,15.0,113013,11,9.500000
9398386,4,9,9,64,0,12,2.0,187880,93,3.791667
12284925,14,99,15,42,0,15,7.0,25090,47,8.133333


### Product features

### User x Product features

## Baseline RMSE

## Create the Pipeline

## Export Model

In [47]:
pipe_reorder = None # NOTE: Temporary

if USE_S3:
    import s3fs
    fs = s3fs.S3FileSystem()
    with fs.open(f's3://{S3_BUCKET}/model2.joblib', 'wb') as f:
        joblib.dump(pipe_reorder, f)
    print(f'Model saved to: s3://{S3_BUCKET}/model2.joblib')
else:
    models_dir = os.path.join(BASE_DIR, '..', 'models')
    os.makedirs(models_dir, exist_ok=True)
    out_path = os.path.join(models_dir, 'model2.joblib')
    joblib.dump(pipe_reorder, out_path)
    print(f'Model saved to: {out_path}')

Model saved to: /home/leo-bellard/nextbuy/notebooks/../models/model2.joblib
